In [ ]:
"""
The Goal of this notebook:
- implement a "take picture mode"
-- action space (send take picture cmd)
-- state -> no change, capture for reward
-- reward change
image quality dependant on movement speed
best case 0 ground track velocity

smear_px = (v_ground * t_exp) / GSD

this means that when the camera bore is essentially focused
on one ground point only this gives a very good image


Verification:
- send take picture cmd periodically in env setup with easy target (coast mode)
- verify reward is generated when it should
- apply

API:
??

use defaults:
limit n pictures to 10 (memory, energy, downlink)
quality rating: 1/[(bore_ground_speed - ground_speed) + e] (e = 1e-3)
-> max quality 1000, worst 0 (infinite speed)
imaging time: 

"""

In [ ]:
from __future__ import annotations
from simulation.state_types import SimulationStateSeries, SimulationTimestepState
import numpy as np


def build_image_quality_state(
    series: SimulationStateSeries,
    frame_idx: int,
    camera_exposure_time: float,
) -> dict:
    """Build sim_state dict for image_quality() from SimulationStateSeries frame."""
    # Orbital speed: omega_rad_s * radius_km
    sat_speed = series.metadata.omega_rad_s * series.radius_km[frame_idx]
    
    # Body rotation rate (placeholder: would come from state if available)
    # For now, use 0 or extract from dynamics if tracking body rate
    sat_omega = 0.0  # TODO: track satellite body rotation rate in SimulationStateSeries
    
    # Camera bore ground intersection (distance from subsatellite point to bore)
    bore_xy = series.camera_ground_center_xy_km[frame_idx]
    sat_xy = np.array([0.0, 0.0])  # in 2D disk coordinates, subsatellite is origin
    distance = np.linalg.norm(bore_xy - sat_xy)
    
    # Camera boresight elevation angle (angle above horizon to bore point)
    # Compute from camera geometry and altitude
    altitude_km = series.sat_altitude_m[frame_idx] / 1000.0
    earth_radius_km = 6371.0  # approximate
    # Slant range from sat to bore ground point
    bore_distance_km = np.linalg.norm(bore_xy)
    # Elevation angle = angle above horizon
    elevation_angle_rad = np.arctan2(altitude_km, bore_distance_km) if bore_distance_km > 0 else 0.0
    
    return {
        "sat_speed": sat_speed,
        "sat_omega": sat_omega,
        "distance_to_camera_bore_collision": distance,
        "camera_exposure_time": camera_exposure_time,
        "ground_sample_distance_at_camera_bore_collision": series.camera_gsd_m[frame_idx],
        "elevation_angle_at_camera_bore_collision": elevation_angle_rad,
    }


def image_quality(
    sim_state: dict,
    numerical_stability_epsilon: float = 1e-9,
) -> float:
    """
    Compute image quality score based on ground motion during exposure.
    
    Quality = 1 / (image_smear + epsilon), where:
    - image_smear = (ground_track_velocity * exposure_time) / GSD
    - ground_track_velocity = orbital_speed + rotational_velocity_component
    
    Args:
        sim_state: Dict with keys: sat_speed, sat_omega, distance_to_camera_bore_collision,
                   camera_exposure_time, ground_sample_distance_at_camera_bore_collision,
                   elevation_angle_at_camera_bore_collision
        numerical_stability_epsilon: Small value to avoid division by zero
    
    Returns:
        Image quality normalized to [0, 1] range
    """
    sat_speed = sim_state["sat_speed"]
    sat_omega = sim_state["sat_omega"]
    distance = sim_state["distance_to_camera_bore_collision"]
    camera_exposure_time = sim_state["camera_exposure_time"]
    gsd = sim_state["ground_sample_distance_at_camera_bore_collision"]
    elevation_angle = sim_state["elevation_angle_at_camera_bore_collision"]

    # Ground-track velocity at camera bore
    # = orbital velocity (parallel to ground at nadir) + rotational component
    # Rotational component: sat_omega * distance * sin(elevation_angle)
    ground_track_velocity_bore = sat_speed + sat_omega * distance * np.sin(elevation_angle)

    def _image_smear(ground_track_velocity: float, t_exp: float, gsd: float) -> float:
        """Pixel displacement during exposure (in GSD units)."""
        return (ground_track_velocity * t_exp) / gsd
    
    # Image smear in GSD units; high smear = low quality
    smear = _image_smear(ground_track_velocity_bore, camera_exposure_time, gsd)
    quality = 1.0 / (smear + numerical_stability_epsilon)

    # Normalize: at 0 smear, quality = 1/epsilon; at infinite smear, quality = 0
    quality_normalized = quality / (1.0 / numerical_stability_epsilon)
    
    return float(np.clip(quality_normalized, 0.0, 1.0))


backend_root=/home/cedric/code/auto-sat-control/backend


In [ ]:
from __future__ import annotations
from sim import CurrentState
from .camera import Camera
import numpy as np

def image_quality(**sim_state: CurrentState,
                camera_index: int,
                numerical_stability_epsilon: float=1e-9,
                )  -> float:
    sat_speed = sim_state["sat_speed"]
    sat_omega = sim_state["sat_omega"]
    distance = sim_state["distance_to_camera_bore_collision"]
    camera_exposure_time = sim_state["camera_exposure_time"]
    gsd = sim_state["ground_sample_distance_at_camera_bore_collision"]


    elevation_angle_at_camera_bore_collision = sim_state["elevation_angle_at_camera_bore_collision"]
    #as a simplification we assume that the orbital velocity is parallel to the ground
    #of course this is wrong at small elevation angles and only true when elevation angle is 90 degrees
    #but it gives us a starting point for the implementation
    #also we assume small fov so velocity at the edge of the image is approximately the same as at the center
    ground_track_velocity_bore = sat_omega * distance * np.sin(elevation_angle_at_camera_bore_collision) + sat_speed

    def _image_smear (ground_track_velocity: float, t_exp: float, GSD: float) -> float:
        return (ground_track_velocity * t_exp) / GSD
    
    #so a high image smear means low quality
    quality = 1/(_image_smear(ground_track_velocity_bore, camera_exposure_time,) + numerical_stability_epsilon)

    #at 0 smear only the numerical stability is left, so quality is 1/epsilon, at infinite smear quality is 0
    quality_normalized = quality / (1/numerical_stability_epsilon)
    
